# Ejercicio Módulo 2
**Inteligencia Artificial - CEIA - FIUBA**

**Santiago Federico Bettig**

En este ejercicio deben implementar un algoritmo de búsqueda que no sea **Búsqueda Primero en Anchura (BFS)** para resolver el problema de la Torre de Hanoi. La nota máxima dependerá del algoritmo implementado:

- **Búsqueda Primero en Profundidad**: nota máxima 6.
- **Búsqueda de Costo Uniforme**: nota máxima 6.
- **Búsqueda de Profundidad Limitada con Profundidad Iterativa**: nota máxima 7.
- **Búsqueda Voraz usando la heurística dada en el aula virtual**: nota máxima 8.
- **Búsqueda Voraz usando una heurística desarrollada por vos**: nota máxima 9.
- **Búsqueda A\* usando la heurística dada en el aula virtual**: nota máxima 9.
- **Búsqueda A\* usando una heurística desarrollada por vos**: nota máxima 10.

La función debe devolver la salida correspondiente a la solución encontrada o `None si no se encontró una solución.

Además, debe calcular métricas de rendimiento que, como mínimo, incluyan:

- `solution_found`: `True` si se encontró la solución, `False` en caso contrario.
- `nodes_explored`: cantidad de nodos explorados (entero).
- `states_visited`: cantidad de estados distintos visitados (entero).
- `nodes_in_frontier`: cantidad de nodos que quedaron en la frontera al finalizar la ejecución (entero).
- `max_depth`: máxima profundidad explorada (entero).
- `cost_total`: costo total para encontrar la solución (float).

In [6]:
from aima_libs.hanoi_states import ProblemHanoi, StatesHanoi
from aima_libs.tree_hanoi import NodeHanoi
from aima_libs.aima import PriorityQueue as AimaPriorityQueue
from queue import PriorityQueue

In [7]:
# Voy a definir cual es la función que voy a utilizar para evaluar
# la prioridad de cada uno de los nodos
# Cada disco tendrá un peso distinto (De 1 a 5) y cada poste 
# también tendrá un peso distinto (0 a 2). A medida que se muevan los discos
# por los postes, la heurística resultante será:

# h(n) = -sumatoria[costo del poste * costo del disco]

# Esto implica que los discos grandes correctamente ubicados tendrán
# mucho mas peso que los discos chicos.

# Ejemplo:
# 5 4 3   | 2 1 | : -[   [(5+4+3)*0] + [(2+1)*1] + [0*2] ] = -3
# 5 4 3 2 |   1 | : -[ [(5+4+3+2)*0] +     [1*1] + [0*2] ] = -1

# A pesar de estar dandole peso al poste del medio, creo que darle prioridad
# superior que al poste inicial es una forma de decirle al agente: "está bien
# mover los dsicos adelante, pero no es el objetivo final"

# Una vez calculada la heurística, tengo que obtener cual es el 
# costo desde la raiz y sumarselo. Esto implica que a caminos
# mas cortos, mas chances tiene ese nodo de ser abordado.

def priority_func(node):
    # Obtengo 3 listas con los discos en cada nodo
    rods = node.state.rods
    # Inicializo mi variable heurística
    heuristic = 0
    # Hago el cálculo de mi heurística
    for index, rod in enumerate(rods):
        heuristic += (sum(rod)*(2-index))
    # Le sumo a la heurística del nodo el costo
    # desde la raiz al nodo.
    heuristic += node.path_cost
    return heuristic

# Voy a definir una función que me permita ejecutar el
# algoritmo.
# Por defecto, se va a ejectuar con 5 discos
def search_algorithm(number_disks=5) -> (NodeHanoi, dict):

    # Voy a crear la lista con enumeración de 
    # discos
    disk_list = [i for i in range(number_disks, 0, -1)]

    # Paso a crear los estados inicial, final y 
    # a inicializar cual es el problema
    initial_state = StatesHanoi(disk_list, [], [], max_disks=number_disks)
    goal_state = StatesHanoi([], [], disk_list, max_disks=number_disks)
    problem = ProblemHanoi(initial=initial_state, goal=goal_state)

    # Lo primero que tenemos que hacer es definir nuestra frontera
    # inicial, es decir, el punto de partida y lo cargaremos 
    # en una lista, que en esta caso funcionará como cola prioritaria
    queue = AimaPriorityQueue(order='min', f=priority_func)
    queue.append(NodeHanoi(problem.initial))

    # Voy a usar un flag para saber cuando se llegó a la solución
    find_solution = False

    # Hay que tener cuidado ya que puede ocurrir que se alcancen nodos
    # ya explorados, por lo que voy a almacenar en una lista los distintos nodos
    # por los cuales pasé 
    explored = set()

    # También voy a contar la cantidad de nodos por los cuales pasé
    n_nodes_exp = 0

    # Voy a registrar cual es el último disco que moví:
    n_last_disk = 0

    # Voy a seguir buscando la solución hasta que no se
    # observe nada en la frontera
    while len(queue) != 0:
        # Saco el nodo del buffer, aumento la cantidad
        # de nodos explorados y lo agrego a la lista de
        # explorados
        priority, node = queue.pop()
        # Hago lo siguiente para salvar la condición en la
        # que el nodo es el raiz:
        if node.action == None:
            n_last_disk = 0
        else:
            # Voy a usar este valor para saber si se mueve
            # dos veces consecutivas el mismo disco (Cosa que no
            # tiene sentido)
            n_last_disk = node.action.disk
            
        n_nodes_exp += 1
        explored.add(node.state)

        # Tengo que revisar si encontré la solución
        # a mi problema
        if problem.goal_test(node.state):
            # De ser así, voy a devolver algunas métricas
            # obtenidas durante la exploración.
            metrics = {
                "solution_found": True, # Se encontró la solución
                "nodes_explored": n_nodes_exp, # Cantidad de nodos explorados
                "states_visited": len(explored), # Cantidad de estados pasados
                "nodes_in_frontier": len(queue), # Longitud de la frontera
                "max_depth": node.depth,    # Máxima profundidad del arbol
                "cost_total": node.state.accumulated_cost, # Costo total acumulado
            }
            solution = node
            return node, metrics

        # Agregamos a la frontera los nodos siguientes que no hayan sido visitados
        for next_node in node.expand(problem):
            if next_node.state not in explored:
                # Ademas, sabemos que no tiene sentido que se mueva dos 
                # veces consecutivas el mismo disco, por lo que voy a bloquear
                # también ese movimiento
                # Es necesario aclarar que si el nodo padre es el raiz, el atributo
                # action tendrá a su parámetro disk en "None" esto es un problema a 
                # la hora de comparar, por lo que tengo que hacer una diferenciación
                if next_node.action == None:
                    actual_disk = 0
                else:
                    actual_disk = next_node.action.disk
                
                if actual_disk is not n_last_disk:
                    queue.append(next_node)


Se prueba la función:

In [8]:
solution, metrics = search_algorithm(number_disks=5)

Veamos las métricas:

In [9]:
for key, value in metrics.items():
    print(f"{key}: {value}")

solution_found: True
nodes_explored: 180
states_visited: 154
nodes_in_frontier: 20
max_depth: 31
cost_total: 31.0


Veamos el camino de estados desde el principio a la solución:

In [10]:
for nodos in solution.path():
    print(nodos)

<Node HanoiState: 5 4 3 2 1 |  | >
<Node HanoiState: 5 4 3 2 |  | 1>
<Node HanoiState: 5 4 3 | 2 | 1>
<Node HanoiState: 5 4 3 | 2 1 | >
<Node HanoiState: 5 4 | 2 1 | 3>
<Node HanoiState: 5 4 1 | 2 | 3>
<Node HanoiState: 5 4 1 |  | 3 2>
<Node HanoiState: 5 4 |  | 3 2 1>
<Node HanoiState: 5 | 4 | 3 2 1>
<Node HanoiState: 5 | 4 1 | 3 2>
<Node HanoiState: 5 2 | 4 1 | 3>
<Node HanoiState: 5 2 1 | 4 | 3>
<Node HanoiState: 5 2 1 | 4 3 | >
<Node HanoiState: 5 2 | 4 3 | 1>
<Node HanoiState: 5 | 4 3 2 | 1>
<Node HanoiState: 5 | 4 3 2 1 | >
<Node HanoiState:  | 4 3 2 1 | 5>
<Node HanoiState: 1 | 4 3 2 | 5>
<Node HanoiState: 1 | 4 3 | 5 2>
<Node HanoiState:  | 4 3 | 5 2 1>
<Node HanoiState: 3 | 4 | 5 2 1>
<Node HanoiState: 3 | 4 1 | 5 2>
<Node HanoiState: 3 2 | 4 1 | 5>
<Node HanoiState: 3 2 1 | 4 | 5>
<Node HanoiState: 3 2 1 |  | 5 4>
<Node HanoiState: 3 2 |  | 5 4 1>
<Node HanoiState: 3 | 2 | 5 4 1>
<Node HanoiState: 3 | 2 1 | 5 4>
<Node HanoiState:  | 2 1 | 5 4 3>
<Node HanoiState: 1 | 2 | 5 4 

Y las acciones que el agente debería aplicar para llegar al objetivo:

In [11]:
for act in solution.solution():
    print(act)

Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 3 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3
Move disk 4 from 1 to 2
Move disk 1 from 3 to 2
Move disk 2 from 3 to 1
Move disk 1 from 2 to 1
Move disk 3 from 3 to 2
Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 5 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3
Move disk 3 from 2 to 1
Move disk 1 from 3 to 2
Move disk 2 from 3 to 1
Move disk 1 from 2 to 1
Move disk 4 from 2 to 3
Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 3 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3
